[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mahsa-agz/mahsa-agz.github.io/blob/main/ds-handbook/exam_prep/day_24/2_sql.ipynb)

# Day 24: SQL (hard)

**How to work:** Read the tables first (columns, one row = what?). Write the query in your head in the order FROM, WHERE, GROUP BY, HAVING, SELECT, ORDER BY. Check the result for duplicates and NULLs before you open the solution.

**Today (28 min):**

| # | Question | Level | Minutes |
|---|---|---|---|
| Q1 | Reporting chain for every employee | hard | 5 |
| Q2 | Revenue of each manager's whole team | hard | 6 |
| Q3 | Fill the missing days, then take the median (review) | hard | 6 |
| Q4 | Biggest referral trees | hard | 6 |
| Q5 | Users who came back after a long break (review) | hard | 5 |

**Handbook:** [SQL](https://mahsa-agz.github.io/ds-handbook/#sql), [cheat sheets](https://mahsa-agz.github.io/ds-handbook/#cheat)

## Tables

**Real tables: Chinook** (a digital music store, 2009 to 2013; MIT license, lerocha/chinook-database):

| Table | One row is | Key columns |
|---|---|---|
| `Customer` | one customer (59) | `CustomerId`, `FirstName`, `LastName`, `Company` (often NULL), `City`, `State` (often NULL), `Country`, `Email`, `SupportRepId` |
| `Employee` | one employee (8) | `EmployeeId`, `FirstName`, `LastName`, `Title`, `ReportsTo` (manager id, NULL for the boss), `HireDate`, `City` |
| `Invoice` | one purchase (412) | `InvoiceId`, `CustomerId`, `InvoiceDate`, `BillingCity`, `BillingCountry`, `Total` |
| `InvoiceLine` | one track on an invoice (2,240) | `InvoiceLineId`, `InvoiceId`, `TrackId`, `UnitPrice`, `Quantity` |
| `Track` | one song or video (3,503) | `TrackId`, `Name`, `AlbumId`, `MediaTypeId`, `GenreId`, `Composer` (can be NULL), `Milliseconds`, `Bytes`, `UnitPrice` |
| `Album` / `Artist` | one album (347) / artist (275) | `AlbumId`, `Title`, `ArtistId` / `ArtistId`, `Name` |
| `Genre` / `MediaType` | one genre (25) / format (5) | `GenreId`, `Name` / `MediaTypeId`, `Name` |
| `Playlist` / `PlaylistTrack` | one playlist (18) / one track in a playlist (8,715) | `PlaylistId`, `Name` / `PlaylistId`, `TrackId` |

**Real tables: Northwind** (a food wholesaler; orders 2012-07 to 2023-10; MIT license, jpwhite3/northwind-SQLite3):

| Table | One row is | Columns |
|---|---|---|
| `nw_orders` | one order (16,282) | `order_id`, `customer_id`, `employee_id`, `order_date` ('YYYY-MM-DD HH:MM:SS'), `shipped_date`, `ship_country` |
| `nw_order_items` | one product line of an order (609,283) | `order_id`, `product_id`, `unit_price`, `quantity`, `discount` (0 to 0.25). Revenue = `unit_price * quantity * (1 - discount)` |
| `nw_products` | one product (77) | `product_id`, `product`, `category_id`, `list_price`, `discontinued` |
| `nw_categories` | one category (8) | `category_id`, `category` |
| `nw_customers` | one customer (93) | `customer_id`, `company`, `city`, `country` |
| `nw_employees` | one employee (9) | `employee_id`, `name`, `title`, `reports_to` (manager, NULL for the boss) |

**Made-up shopping-app tables** (invented for practice, generated with a fixed seed: 1,500 users signed up
2024-01-01 to 2024-02-29, activity until 2024-03-31):

| Table | One row is | Columns |
|---|---|---|
| `app_users` | one user | `user_id`, `signup_date` ('YYYY-MM-DD'), `country` (US/KR/BR/ID/DE), `platform` (ios/android/web), `channel` (organic/paid_social/search/referral), `referred_by` (user_id of the referrer, NULL unless channel = referral) |
| `daily_activity` | one user on one day they opened the app | `user_id`, `activity_date` ('YYYY-MM-DD'), `sessions`, `minutes`. No row = not active that day |
| `app_events` | one shopping event | `event_id`, `user_id`, `event_time` ('YYYY-MM-DD HH:MM:SS'), `event_name` (view_item / add_to_cart / checkout / purchase) |
| `purchases` | one order | `order_id`, `user_id`, `order_time`, `amount` (USD), `status` (completed / refunded) |
| `app_events_raw` | one event as received from the apps (has retries and double taps) | `raw_id`, `user_id`, `event_time`, `event_name`, `received_at` |

**Real table: Online Retail** (UCI, a UK online gift shop, 2010-12-01 to 2011-12-09, CC BY 4.0):

| Table | One row is | Columns |
|---|---|---|
| `retail` | one product line of an invoice (541,909) | `invoice_no` (text; a 'C' prefix = cancellation), `stock_code`, `description`, `quantity` (negative for cancellations), `invoice_date` ('YYYY-MM-DD HH:MM:SS'), `unit_price` (GBP), `customer_id` (NULL for about 25% of rows), `country` |

The setup cell prints every table with its columns. Call `tables()` again any time.

## Setup

Run this cell once before the questions.

In [ ]:
import sqlite3
import pandas as pd

def run(q, con=None):
    """Runs a query and shows the result as a table (SELECT / WITH); other statements just run."""
    con = con or db
    if not q.strip():
        print("Write your query between the triple quotes, then run the cell.")
        return None
    if q.strip().lower().startswith(("select", "with")):
        return pd.read_sql_query(q, con)
    con.executescript(q)
    print("done")

# Finds the data folder (local copy of the repository, or /content/data in Colab) and downloads a missing file
# from the ds-handbook repository on GitHub (or from its original source).
import os, sys, urllib.request

RAW = "https://raw.githubusercontent.com/mahsa-agz/mahsa-agz.github.io/main/ds-handbook/exam_prep/data/"

def _find_data():
    if "google.colab" in sys.modules:
        os.makedirs("/content/data", exist_ok=True)
        return "/content/data"
    for p in ("../data", "data", "exam_prep/data"):
        if os.path.isdir(p):
            return p
    os.makedirs("data", exist_ok=True)
    return "data"

DATA = _find_data()
print("data folder:", DATA)

def data_path(name, url=None):
    """Local path of a data file; downloads it from the repository (or from url) if it is not there yet."""
    path = os.path.join(DATA, name)
    if not os.path.exists(path):
        for source in (RAW + name, url):
            if not source:
                continue
            try:
                print("downloading", name)
                urllib.request.urlretrieve(source, path)
                break
            except Exception as e:
                print("  could not download from", source, "->", e)
    return path

db = sqlite3.connect(':memory:')

# Real data: the Chinook digital music store (copied into the in-memory db).
_src = sqlite3.connect(data_path("chinook.sqlite", "https://github.com/lerocha/chinook-database/releases/download/v1.4.5/Chinook_Sqlite.sqlite"))
_src.backup(db)
_src.close()

# Real data: Northwind orders (jpwhite3/northwind-SQLite3, MIT). Copied into db with short table names.
db.execute("ATTACH DATABASE ? AS nw", (data_path("northwind.db", "https://raw.githubusercontent.com/jpwhite3/northwind-SQLite3/main/dist/northwind.db"),))
db.executescript("""
CREATE TABLE nw_customers  AS SELECT CustomerID AS customer_id, CompanyName AS company, City AS city, Country AS country FROM nw.Customers;
CREATE TABLE nw_employees  AS SELECT EmployeeID AS employee_id, FirstName || ' ' || LastName AS name, Title AS title, ReportsTo AS reports_to FROM nw.Employees;
CREATE TABLE nw_categories AS SELECT CategoryID AS category_id, CategoryName AS category FROM nw.Categories;
CREATE TABLE nw_products   AS SELECT ProductID AS product_id, ProductName AS product, CategoryID AS category_id, UnitPrice AS list_price, Discontinued AS discontinued FROM nw.Products;
CREATE TABLE nw_orders     AS SELECT OrderID AS order_id, CustomerID AS customer_id, EmployeeID AS employee_id, OrderDate AS order_date, ShippedDate AS shipped_date, ShipCountry AS ship_country FROM nw.Orders;
CREATE TABLE nw_order_items AS SELECT OrderID AS order_id, ProductID AS product_id, UnitPrice AS unit_price, Quantity AS quantity, Discount AS discount FROM nw."Order Details";
""")
db.execute("DETACH DATABASE nw")

# Made-up shopping-app tables (generated with a fixed seed, so everyone gets the same rows).
import numpy as np

def make_app_tables(con, seed=7):
    rs = np.random.RandomState(seed)
    start, end = pd.Timestamp("2024-01-01"), pd.Timestamp("2024-03-31")
    n = 1500
    countries, c_p = ["US", "KR", "BR", "ID", "DE"], [0.30, 0.25, 0.20, 0.15, 0.10]
    platforms, p_p = ["ios", "android", "web"], [0.40, 0.45, 0.15]
    channels, ch_p = ["organic", "paid_social", "search", "referral"], [0.35, 0.30, 0.20, 0.15]
    signup_off = np.sort(rs.randint(0, 60, n))                     # day offsets 0..59 (Jan 1 to Feb 29)
    country = rs.choice(len(countries), n, p=c_p)
    platform = rs.choice(len(platforms), n, p=p_p)
    channel = rs.choice(len(channels), n, p=ch_p)
    referred_by = [None] * n
    for i in range(n):
        if channels[channel[i]] == "referral":
            earlier = np.nonzero(signup_off[:i] < signup_off[i])[0]
            if len(earlier):
                referred_by[i] = int(earlier[rs.randint(len(earlier))]) + 1
            else:
                channel[i] = 0                                       # nobody to refer them: organic
    users = pd.DataFrame({
        "user_id": np.arange(1, n + 1),
        "signup_date": [(start + pd.Timedelta(days=int(d))).strftime("%Y-%m-%d") for d in signup_off],
        "country": [countries[c] for c in country],
        "platform": [platforms[p] for p in platform],
        "channel": [channels[c] for c in channel],
        "referred_by": pd.array(referred_by, dtype="Int64"),
    })
    plat_boost = {"ios": 0.08, "android": 0.0, "web": -0.10}
    chan_boost = {"organic": 0.05, "paid_social": -0.08, "search": 0.0, "referral": 0.08}
    act, ev, buys = [], [], []
    total_days = (end - start).days + 1
    for i in range(n):
        uid = i + 1
        p = rs.beta(2, 3) + plat_boost[platforms[platform[i]]] + chan_boost[channels[channel[i]]]
        p = min(max(p, 0.03), 0.9)
        life = 1 if rs.rand() < 0.30 else 1 + int(rs.exponential(35))   # days until the user stops coming back
        buyer = rs.rand() < 0.55
        prev = True
        for t in range(0, total_days - signup_off[i]):
            if t == 0:
                active = True
            elif t >= life:
                active = rs.rand() < 0.01                                 # rare comeback after churn
            else:
                active = rs.rand() < (min(p + 0.25, 0.95) if prev else p * 0.7)
            prev = active
            if not active:
                continue
            day = start + pd.Timedelta(days=int(signup_off[i] + t))
            sessions = 1 + rs.poisson(0.8)
            minutes = int(sum(rs.randint(2, 25) for _ in range(sessions)))
            act.append((uid, day.strftime("%Y-%m-%d"), sessions, minutes))
            clock = day + pd.Timedelta(seconds=int(rs.randint(6 * 3600, 22 * 3600)))
            if rs.rand() < 0.65:
                steps = ["view_item"]
                if rs.rand() < 0.40:
                    steps.append("add_to_cart")
                    if rs.rand() < 0.55:
                        steps.append("checkout")
                        if buyer and rs.rand() < 0.75:
                            steps.append("purchase")
                elif rs.rand() < 0.03:
                    steps = ["add_to_cart"]                               # from the wishlist, no view that day
                for s in steps:
                    clock = clock + pd.Timedelta(seconds=int(rs.randint(15, 900)))
                    ev.append((uid, clock.strftime("%Y-%m-%d %H:%M:%S"), s))
                    if s == "purchase":
                        amount = round(float(rs.gamma(2.0, 18.0)) + 5, 2)
                        status = "refunded" if rs.rand() < 0.07 else "completed"
                        buys.append((uid, clock.strftime("%Y-%m-%d %H:%M:%S"), amount, status))
    activity = pd.DataFrame(act, columns=["user_id", "activity_date", "sessions", "minutes"])
    events = pd.DataFrame(ev, columns=["user_id", "event_time", "event_name"])
    events = events.sort_values(["event_time", "user_id"], kind="mergesort").reset_index(drop=True)
    events.insert(0, "event_id", np.arange(1, len(events) + 1))
    purchases = pd.DataFrame(buys, columns=["user_id", "order_time", "amount", "status"])
    purchases = purchases.sort_values(["order_time", "user_id"], kind="mergesort").reset_index(drop=True)
    purchases.insert(0, "order_id", np.arange(1, len(purchases) + 1))
    # raw event log as it arrives from the apps: retries create exact copies, double taps create near copies
    raw = events.copy()
    dup = raw[rs.rand(len(raw)) < 0.04]
    tap = raw[rs.rand(len(raw)) < 0.01].copy()
    tap["event_time"] = (pd.to_datetime(tap["event_time"]) +
                         pd.to_timedelta(rs.randint(1, 4, len(tap)), unit="s")).dt.strftime("%Y-%m-%d %H:%M:%S")
    raw = pd.concat([raw, dup, tap]).sort_values(["event_time", "user_id"], kind="mergesort").reset_index(drop=True)
    raw = raw.drop(columns="event_id")
    raw.insert(0, "raw_id", np.arange(1, len(raw) + 1))
    raw["received_at"] = (pd.to_datetime(raw["event_time"]) +
                          pd.to_timedelta(rs.randint(1, 600, len(raw)), unit="s")).dt.strftime("%Y-%m-%d %H:%M:%S")
    for name, df in [("app_users", users), ("daily_activity", activity), ("app_events", events),
                     ("purchases", purchases), ("app_events_raw", raw)]:
        df.to_sql(name, con, index=False, if_exists="replace")

make_app_tables(db)

# Real data: UCI Online Retail (a UK gift shop, 2010-12 to 2011-12, CC BY 4.0).
import io, zipfile
_p = os.path.join(DATA, "online_retail.csv.gz")
if not os.path.exists(_p):                       # first run in Colab: download the UCI zip and convert it once
    with zipfile.ZipFile(data_path("online_retail.zip", "https://archive.ics.uci.edu/static/public/352/online+retail.zip")) as zf:
        pd.read_excel(io.BytesIO(zf.read("Online Retail.xlsx"))).to_csv(_p, index=False, compression="gzip")
_o = pd.read_csv(_p, dtype={"InvoiceNo": str, "StockCode": str})
_o.columns = ["invoice_no", "stock_code", "description", "quantity", "invoice_date", "unit_price", "customer_id", "country"]
_o["invoice_date"] = pd.to_datetime(_o["invoice_date"]).dt.strftime("%Y-%m-%d %H:%M:%S")
_o["customer_id"] = _o["customer_id"].astype("Int64")
_o.to_sql("retail", db, index=False)

def tables():
    """Prints every table with its row count and columns."""
    for (t,) in db.execute("SELECT name FROM sqlite_master WHERE type = 'table' ORDER BY name"):
        cols = [r[1] for r in db.execute(f'PRAGMA table_info("{t}")')]
        n = db.execute(f'SELECT COUNT(*) FROM "{t}"').fetchone()[0]
        print(f"{t} ({n:,} rows): {', '.join(cols)}")

tables()

---
### Q1. Reporting chain for every employee

*hard, about 5 min*

Chinook (real) `Employee` has `ReportsTo` (the manager's EmployeeId; NULL for the top person). Return every
employee with `EmployeeId`, `name` (first + last), `Title`, `level` (0 for the top person, 1 for their direct
reports, ...) and `path`: last names from the top down, joined with ' > '. Order by `path`.

Example: if Adams manages Edwards and Edwards manages Park, Park's row is `level 2, path 'Adams > Edwards > Park'`.

*Follow-up:* the HR table has a data bug: A reports to B and B reports to A. What happens to your query, and how do you protect it?

In [ ]:
run("""

""")

<details><summary><b>Hint 1</b></summary>

Signal: a hierarchy of unknown depth. Pattern: recursive CTE: an anchor (the root) UNION ALL a step that joins children to the rows found so far.

</details>

<details><summary><b>Hint 2</b></summary>

1. `WITH RECURSIVE chain(...) AS (`
2. Anchor: `SELECT ..., 0, LastName FROM Employee WHERE ReportsTo IS NULL`.
3. Step: `SELECT e..., c.level + 1, c.path || ' > ' || e.LastName FROM Employee e JOIN chain c ON e.ReportsTo = c.EmployeeId`.
4. Close the CTE, SELECT from chain ORDER BY path.

</details>

<details><summary><b>Solution</b></summary>

```sql
WITH RECURSIVE chain(EmployeeId, name, Title, level, path) AS (
  SELECT EmployeeId, FirstName || ' ' || LastName, Title, 0, LastName
  FROM Employee
  WHERE ReportsTo IS NULL
  UNION ALL
  SELECT e.EmployeeId, e.FirstName || ' ' || e.LastName, e.Title,
         c.level + 1, c.path || ' > ' || e.LastName
  FROM Employee e
  JOIN chain c ON e.ReportsTo = c.EmployeeId
)
SELECT EmployeeId, name, Title, level, path
FROM chain
ORDER BY path
```

Result:

| EmployeeId | name | Title | level | path |
|---|---|---|---|---|
| 1 | Andrew Adams | General Manager | 0 | Adams |
| 2 | Nancy Edwards | Sales Manager | 1 | Adams > Edwards |
| 5 | Steve Johnson | Sales Support Agent | 2 | Adams > Edwards > Johnson |
| 4 | Margaret Park | Sales Support Agent | 2 | Adams > Edwards > Park |
| 3 | Jane Peacock | Sales Support Agent | 2 | Adams > Edwards > Peacock |
| 6 | Michael Mitchell | IT Manager | 1 | Adams > Mitchell |
| 8 | Laura Callahan | IT Staff | 2 | Adams > Mitchell > Callahan |
| 7 | Robert King | IT Staff | 2 | Adams > Mitchell > King |

**Why:** The anchor returns the root; each pass of the recursive part adds the next level by joining employees whose manager is already in `chain`; it stops when a pass adds no rows. The path string is built on the way down. Andrew Adams is level 0; the three Sales Support Agents are level 2 under Edwards. Follow-up: with a cycle and no root, the anchor finds nothing for those two rows (they never appear); a cycle BELOW a root loops forever. Protect with a depth limit (`WHERE c.level < 20`) or a visited check (`WHERE c.path NOT LIKE '%' || e.LastName || '%'`, better on ids); PostgreSQL 14 also has a `CYCLE` clause.

**Complexity:** O(n) rows produced for a tree; one join per level.

**Common mistakes:** Forgetting the RECURSIVE keyword (needed in PostgreSQL and SQLite style). UNION instead of UNION ALL (extra dedup work; also hides bugs). Starting from the leaves and walking up when the question asks top down.

**Learn more:** [sql-recursive-cte](https://mahsa-agz.github.io/ds-handbook/#sql-recursive-cte), [sql-self-join](https://mahsa-agz.github.io/ds-handbook/#sql-self-join)

</details>

---
### Q2. Revenue of each manager's whole team

*hard, about 6 min*

Northwind (real). Revenue = `unit_price * quantity * (1 - discount)`, orders in **2022**, credited to the
order's `employee_id`. For **every** employee return `name`, `title`, `reports_all` (direct AND indirect reports)
and `team_revenue_2022`: revenue of the employee plus everyone below them (rounded), plus `pct_of_company` (1 decimal).
Order by team revenue descending.

Example: if Fuller manages Buchanan, who manages Suyama, Fuller's team revenue includes Suyama's orders.

In [ ]:
run("""

""")

<details><summary><b>Hint 1</b></summary>

Signal: 'direct and indirect' = the whole subtree. Pattern: recursive CTE that builds (manager, member) pairs, then a join to revenue and GROUP BY manager.

</details>

<details><summary><b>Hint 2</b></summary>

1. Anchor: every employee is in their own team: `SELECT employee_id, employee_id FROM nw_employees`.
2. Step: `SELECT s.manager_id, e.employee_id FROM sub s JOIN nw_employees e ON e.reports_to = s.employee_id`.
3. CTE `rev`: 2022 revenue per employee.
4. `sub LEFT JOIN rev`, GROUP BY manager; `COUNT(*) - 1` = reports.

</details>

<details><summary><b>Solution</b></summary>

```sql
WITH RECURSIVE sub(manager_id, employee_id) AS (
  SELECT employee_id, employee_id FROM nw_employees
  UNION ALL
  SELECT s.manager_id, e.employee_id
  FROM sub s
  JOIN nw_employees e ON e.reports_to = s.employee_id
), rev AS (
  SELECT o.employee_id, SUM(i.unit_price * i.quantity * (1 - i.discount)) AS revenue
  FROM nw_orders o
  JOIN nw_order_items i ON i.order_id = o.order_id
  WHERE o.order_date >= '2022-01-01' AND o.order_date < '2023-01-01'
  GROUP BY o.employee_id
)
SELECT m.name, m.title, COUNT(*) - 1 AS reports_all,
  ROUND(SUM(r.revenue)) AS team_revenue_2022,
  ROUND(100.0 * SUM(r.revenue) / (SELECT SUM(revenue) FROM rev), 1) AS pct_of_company
FROM sub s
JOIN nw_employees m ON m.employee_id = s.manager_id
LEFT JOIN rev r ON r.employee_id = s.employee_id
GROUP BY s.manager_id, m.name, m.title
ORDER BY team_revenue_2022 DESC
```

Result:

| name | title | reports_all | team_revenue_2022 | pct_of_company |
|---|---|---|---|---|
| Andrew Fuller | Vice President, Sales | 8 | 39742066.0 | 100.0 |
| Steven Buchanan | Sales Manager | 3 | 18712055.0 | 47.1 |
| Robert King | Sales Representative | 0 | 5031166.0 | 12.7 |
| Anne Dodsworth | Sales Representative | 0 | 4664363.0 | 11.7 |
| Margaret Peacock | Sales Representative | 0 | 4420627.0 | 11.1 |
| Michael Suyama | Sales Representative | 0 | 4344931.0 | 10.9 |
| Nancy Davolio | Sales Representative | 0 | 4300391.0 | 10.8 |
| Laura Callahan | Inside Sales Coordinator | 0 | 3926407.0 | 9.9 |
| Janet Leverling | Sales Representative | 0 | 3844719.0 | 9.7 |

**Why:** Starting the recursion from EVERY employee (not only the root) gives the full subtree of each person in one query: the pair (manager, member) exists once for every member at any depth. The VP (Andrew Fuller) has all 8 others below him, so his team is 100% of 2022 revenue; Steven Buchanan's team of 3 makes 47.1%. Pre-aggregating revenue per employee BEFORE joining the tree keeps the join small.

**Complexity:** Pairs = sum of subtree sizes (O(n * depth)); revenue is aggregated once.

**Common mistakes:** Only direct reports (a self-join, not recursion). Joining 600k order lines to the tree before aggregating (slow, same answer). `COUNT(*)` as reports (it includes the person: subtract 1).

**Learn more:** [sql-recursive-cte](https://mahsa-agz.github.io/ds-handbook/#sql-recursive-cte), [sql-aggregation](https://mahsa-agz.github.io/ds-handbook/#sql-aggregation)

</details>

---
### Q3. Fill the missing days, then take the median (review)

*hard, about 6 min*

UCI Online Retail (real). Daily revenue = `SUM(quantity * unit_price)` of real sales (`invoice_no` not
starting with 'C', `quantity > 0`, `unit_price > 0`) by calendar day of `invoice_date`. Days with no sales do not appear
in the table at all. For **2010-12-01 to 2011-02-28** (90 days) return one row: `days`, `sales_days`,
`median_all_days` (zero-revenue days included), `median_sales_days`, `mean_all_days`, `mean_sales_days` (all rounded
to whole numbers).

*Follow-up:* the shop is closed on Saturdays and over Christmas. Which median would you put in a forecast, and why?

In [ ]:
run("""

""")

<details><summary><b>Hint 1</b></summary>

Signal: 'days with no rows' must count as zero. Pattern: a date spine from a recursive CTE, LEFT JOIN the data, COALESCE to 0; then medians with ROW_NUMBER / COUNT.

</details>

<details><summary><b>Hint 2</b></summary>

1. `WITH RECURSIVE days(d) AS (SELECT '2010-12-01' UNION ALL SELECT date(d, '+1 day') FROM days WHERE d < '2011-02-28')`.
2. `sales`: revenue per `substr(invoice_date, 1, 10)`.
3. `daily`: days LEFT JOIN sales, COALESCE(revenue, 0).
4. Two ranked CTEs (all days / days with revenue > 0) and scalar subqueries for each number.

</details>

<details><summary><b>Solution</b></summary>

```sql
WITH RECURSIVE days(d) AS (
  SELECT '2010-12-01'
  UNION ALL
  SELECT date(d, '+1 day') FROM days WHERE d < '2011-02-28'
), sales AS (
  SELECT substr(invoice_date, 1, 10) AS d, SUM(quantity * unit_price) AS revenue
  FROM retail
  WHERE invoice_no NOT LIKE 'C%' AND quantity > 0 AND unit_price > 0
  GROUP BY substr(invoice_date, 1, 10)
), daily AS (
  SELECT days.d, COALESCE(s.revenue, 0) AS revenue
  FROM days LEFT JOIN sales s ON s.d = days.d
), r AS (
  SELECT revenue, ROW_NUMBER() OVER (ORDER BY revenue) AS rn, COUNT(*) OVER () AS cnt
  FROM daily
), s AS (
  SELECT revenue, ROW_NUMBER() OVER (ORDER BY revenue) AS rn, COUNT(*) OVER () AS cnt
  FROM daily WHERE revenue > 0
)
SELECT
  (SELECT COUNT(*) FROM daily) AS days,
  (SELECT COUNT(*) FROM daily WHERE revenue > 0) AS sales_days,
  (SELECT ROUND(AVG(revenue)) FROM r WHERE rn IN ((cnt + 1) / 2, (cnt + 2) / 2)) AS median_all_days,
  (SELECT ROUND(AVG(revenue)) FROM s WHERE rn IN ((cnt + 1) / 2, (cnt + 2) / 2)) AS median_sales_days,
  (SELECT ROUND(AVG(revenue)) FROM daily) AS mean_all_days,
  (SELECT ROUND(AVG(revenue)) FROM daily WHERE revenue > 0) AS mean_sales_days
```

Result:

| days | sales_days | median_all_days | median_sales_days | mean_all_days | mean_sales_days |
|---|---|---|---|---|---|
| 90 | 68 | 21501.0 | 25834.0 | 22653.0 | 29982.0 |

**Why:** Without the spine, the 22 closed days simply do not exist and every daily statistic is computed on 68 days only: the mean jumps from 22,653 to 29,982. The recursive CTE generates one row per calendar day, and the LEFT JOIN keeps days without sales. Follow-up: for a calendar forecast (how much per day next quarter) use all days (median 21,501); to describe a typical trading day use sales days (median 25,834). Saying which denominator you chose is the point.

**Complexity:** The spine is 90 rows; the sales aggregation is one scan.

**Common mistakes:** Joining the spine with `date(invoice_date) = d` on 500k rows without pre-aggregating (slow). Forgetting COALESCE (NULLs are skipped by AVG and the median, so zeros vanish again). PostgreSQL has `generate_series('2010-12-01'::date, '2011-02-28', interval '1 day')` instead of the recursive CTE.

**Learn more:** [sql-recursive-cte](https://mahsa-agz.github.io/ds-handbook/#sql-recursive-cte), [sql-median-percentiles](https://mahsa-agz.github.io/ds-handbook/#sql-median-percentiles), [sql-dates](https://mahsa-agz.github.io/ds-handbook/#sql-dates)

</details>

---
### Q4. Biggest referral trees

*hard, about 6 min*

Shopping app (made up). `app_users.referred_by` is the user who invited this user (NULL if nobody did).
A **root** is a user with `referred_by IS NULL`. Everyone invited by a root, by someone they invited, and so on,
belongs to the root's tree. Return the 5 roots with the largest trees: `root_id`, `referred_total` (everyone in the
tree except the root), `direct` (invited by the root personally) and `max_depth` (1 = direct invites only). Ties: lower
`root_id` first.

Example: 2 invites 10 and 11; 10 invites 50: root 2 has referred_total 3, direct 2, max_depth 2.

*Follow-up:* growth wants to reward "super referrers". Is `referred_total` or `direct` the fairer metric?

In [ ]:
run("""

""")

<details><summary><b>Hint 1</b></summary>

Signal: invites of invites of invites. Pattern: recursive CTE that carries the root id and the depth down the tree.

</details>

<details><summary><b>Hint 2</b></summary>

1. Anchor: `SELECT user_id AS root_id, user_id, 0 AS depth FROM app_users WHERE referred_by IS NULL`.
2. Step: join `app_users u ON u.referred_by = t.user_id`, keep `t.root_id`, depth + 1.
3. GROUP BY root_id: COUNT(*) - 1, SUM(depth = 1), MAX(depth).

</details>

<details><summary><b>Solution</b></summary>

```sql
WITH RECURSIVE tree(root_id, user_id, depth) AS (
  SELECT user_id, user_id, 0
  FROM app_users
  WHERE referred_by IS NULL
  UNION ALL
  SELECT t.root_id, u.user_id, t.depth + 1
  FROM tree t
  JOIN app_users u ON u.referred_by = t.user_id
)
SELECT root_id,
  COUNT(*) - 1 AS referred_total,
  SUM(CASE WHEN depth = 1 THEN 1 ELSE 0 END) AS direct,
  MAX(depth) AS max_depth
FROM tree
GROUP BY root_id
ORDER BY referred_total DESC, root_id
LIMIT 5
```

Result:

| root_id | referred_total | direct | max_depth |
|---|---|---|---|
| 2 | 8 | 2 | 3 |
| 73 | 5 | 1 | 2 |
| 123 | 5 | 4 | 2 |
| 18 | 4 | 3 | 2 |
| 145 | 4 | 3 | 2 |

**Why:** Carrying `root_id` unchanged through the recursion labels every descendant with its tree. All 1,500 users land in exactly one of 1,273 trees (227 users were referred), and the deepest chain is 3 levels. User 2 leads with 8 people in the tree but only 2 direct invites. Follow-up: `direct` measures the user's own action; `referred_total` rewards being early (old users have had more time for their invitees to invite). Use direct invites for rewards, and the tree size to study virality (k-factor per level).

**Complexity:** Every user is visited once: O(n) rows, one join per level.

**Common mistakes:** Self-joining a fixed number of times (misses deeper levels). Starting from all users instead of roots (each subtree is counted several times). No cycle guard for real referral data (add a depth limit).

**Learn more:** [sql-recursive-cte](https://mahsa-agz.github.io/ds-handbook/#sql-recursive-cte)

</details>

---
### Q5. Users who came back after a long break (review)

*hard, about 5 min*

Shopping app (made up), `daily_activity`. A **comeback** is an active day that follows 14 or more inactive
days in a row (e.g. active Jan 1, next active Jan 16: 14 inactive days, a comeback). Per `platform` return `users`
(users with any activity), `resurrected` (users with at least one comeback), `resurrected_pct` (1 decimal) and
`longest_gap` (the longest run of inactive days seen on that platform, as an integer). Order by platform.

In [ ]:
run("""

""")

<details><summary><b>Hint 1</b></summary>

Signal: the gap between one active day and the next. Pattern: gaps and islands with LAG: gap = days between consecutive rows minus 1.

</details>

<details><summary><b>Hint 2</b></summary>

1. `julianday(d) - julianday(LAG(d) OVER (PARTITION BY user_id ORDER BY d)) - 1` = inactive days before d.
2. Per user: MAX(gap), SUM(gap >= 14).
3. Join platform, aggregate.

</details>

<details><summary><b>Solution</b></summary>

```sql
WITH g AS (
  SELECT user_id, activity_date,
    CAST(julianday(activity_date)
      - julianday(LAG(activity_date) OVER (PARTITION BY user_id ORDER BY activity_date)) - 1 AS INTEGER) AS gap_days
  FROM daily_activity
), per_user AS (
  SELECT user_id, MAX(gap_days) AS longest_gap,
    SUM(CASE WHEN gap_days >= 14 THEN 1 ELSE 0 END) AS comebacks
  FROM g
  GROUP BY user_id
)
SELECT u.platform, COUNT(*) AS users,
  SUM(CASE WHEN p.comebacks > 0 THEN 1 ELSE 0 END) AS resurrected,
  ROUND(100.0 * SUM(CASE WHEN p.comebacks > 0 THEN 1 ELSE 0 END) / COUNT(*), 1) AS resurrected_pct,
  MAX(p.longest_gap) AS longest_gap
FROM per_user p
JOIN app_users u ON u.user_id = p.user_id
GROUP BY u.platform
ORDER BY u.platform
```

Result:

| platform | users | resurrected | resurrected_pct | longest_gap |
|---|---|---|---|---|
| android | 695 | 201 | 28.9 | 75 |
| ios | 594 | 159 | 26.8 | 78 |
| web | 211 | 60 | 28.4 | 68 |

**Why:** LAG gives the previous active day of the same user; the difference minus 1 is the number of inactive days in between (Jan 1 to Jan 16 is 15 days apart, 14 inactive). The first row of each user has a NULL gap, which the CASE and MAX ignore. About 27% to 29% of users on every platform come back after a 2 week break, so '14 days inactive = churned' would mislabel many users.

**Complexity:** One window sort per user: O(n log n).

**Common mistakes:** Forgetting the minus 1 (off by one: a 13 day gap is counted). Counting comebacks instead of users. Ignoring the gap at the END (a user inactive since their last day is churned, not resurrected; that is a different question).

**Learn more:** [sql-gaps-islands](https://mahsa-agz.github.io/ds-handbook/#sql-gaps-islands), [sql-lag-lead](https://mahsa-agz.github.io/ds-handbook/#sql-lag-lead)

</details>

---
## Redo list

Write down every question you got wrong, solved only after Hint 2, or finished over time. Solve it again from scratch 3 days and 7 days later.

**Next:** [Statistics, A/B testing and product](https://colab.research.google.com/github/mahsa-agz/mahsa-agz.github.io/blob/main/ds-handbook/exam_prep/day_24/3_stats.ipynb)